In [ ]:
!pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 710.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 6.9 MB/s eta 0:00:00


In [ ]:
!pip install state

  Preparing metadata (setup.py) ... done
  Created wheel for state: filename=state-0.1.1.dev0-py3-none-any.whl size=1726 sha256=5c036e505c635bd940230b8a7bb1b786769d34eba3ffafb13010d44eba5e3161
  Stored in directory: /root/.cache/pip/wheels/ee/42/67/f25ca748a07e05efbc004bb6535f0f906e09714b4fb7307448
Successfully built state


In [ ]:
import json
import cohere
from langgraph.graph import StateGraph

In [ ]:
l = []

In [ ]:
EMPLOYEE_DB = {
    "EMP001": {
        "employee_name": "Akash",
        "status": "Active",
        "grade": "L2",
        "department": "Engineering"
    }
}


def employee_lookup(employee_id: str):
  if employee_id not in EMPLOYEE_DB.keys():
    return "END"
  else:
    return EMPLOYEE_DB.get(employee_id)

In [ ]:
TRAVEL_DB = {
    "TR1001": {
        "status": "Approved",
        "purpose": "Client Meeting",
        "destination": "Mumbai"
    }
}


def travel_lookup(travel_id: str):
  return TRAVEL_DB.get(travel_id)

In [ ]:
from typing import TypedDict, Dict, Any


class ClaimState(TypedDict):
    claim: Dict[str, Any]

    enterprise_validation: Dict[str, Any]

    receipt_validation: Dict[str, Any]

    claim_validation: Dict[str, Any]

    policy_report: Dict[str, Any]

    final_decision: Dict[str, Any]

In [ ]:
SYSTEM_PROMPT = """
You are an Enterprise Validation Agent.

Responsibilities:

1. Understand the reimbursement claim.
2. Validate business context using tool outputs.
3. Produce an enterprise validation summary.

Never:

- Approve claim
- Reject claim
- Apply reimbursement policy
- Calculate reimbursement

Return ONLY valid JSON.
"""

In [ ]:
co = cohere.ClientV2(
    api_key="FttHeK9M6nPNjvZHw7pqOuoB6fok7CP50DeEe4Mr"
)


def enterprise_validation_node(
    state: ClaimState
):

    claim = state["claim"]

    employee = employee_lookup(
        claim["employee_id"]
    )
    if employee == "END":
      return "END"

    travel = travel_lookup(
        claim["travel_request_id"]
    )

    context = {
        "claim": claim,
        "employee": employee,
        "travel": travel
    }

    response = co.chat(
        model="command-a-03-2025",

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(context)
            }
        ],

        response_format={
            "type": "json_object"
        }
    )

    output = json.loads(
        response.message.content[0].text
    )
    l.append(output)
    return {
        # **state,

        "enterprise_validation": output
    }

In [ ]:
# builder = StateGraph(ClaimState)

# builder.add_node(
#     "enterprise_validation",
#     enterprise_validation_node
# )

# builder.set_entry_point(
#     "enterprise_validation"
# )

# graph = builder.compile()

In [ ]:
# claim = {
#     "claim": {
#         "claim_id": "CLM1001",
#         "employee_id": "EMP001",
#         "travel_request_id": "TR1001",
#         "business_purpose": "Client Meeting",
#         "expenses": [
#             {
#                 "expense_type": "Hotel",
#                 "amount": 6500
#             }
#         ]
#     }
# }


# result = graph.invoke(
#     claim
# )

In [ ]:
result

{'claim': {'claim_id': 'CLM1001',
  'employee_id': 'EMP001',
  'travel_request_id': 'TR1001',
  'business_purpose': 'Client Meeting',
  'expenses': [{'expense_type': 'Hotel', 'amount': 6500}]},
 'enterprise_validation': {'claim_id': 'CLM1001',
  'employee_id': 'EMP001',
  'employee_name': 'Akash',
  'employee_status': 'Active',
  'employee_grade': 'L2',
  'employee_department': 'Engineering',
  'travel_request_id': 'TR1001',
  'travel_status': 'Approved',
  'travel_purpose': 'Client Meeting',
  'travel_destination': 'Mumbai',
  'claim_business_purpose': 'Client Meeting',
  'expenses': [{'expense_type': 'Hotel', 'amount': 6500}]}}

In [ ]:
############################# 2ND agent########################

In [ ]:
def ocr_tool(claim: dict):
    # print("claim",claim)

    submitted_date = claim['claim']["submitted_date"]
    expenses = claim['claim']["expenses"]

    # print("expenses === ",expenses)

    MOCK_RECEIPTS = {
        "Hotel": {
            "vendor": "Taj Hotel Mumbai",
            "invoice_number": "HTL-1001",
            "amount": 6500,
            "currency": "INR"
        },
        "Meal": {
            "vendor": "Barbeque Nation",
            "invoice_number": "ML-2001",
            "amount": 850,
            "currency": "INR"
        },
        "Taxi": {
            "vendor": "Uber",
            "invoice_number": "TX-3001",
            "amount": 450,
            "currency": "INR"
        }
    }

    extracted = []

    for expense in expenses:

        receipt = MOCK_RECEIPTS.get(expense["expense_type"])

        if receipt is None:
            continue

        extracted.append({
            "expense_type": expense["expense_type"],
            "receipt_file": expense["receipt_file"],
            "vendor": receipt["vendor"],
            "invoice_number": receipt["invoice_number"],
            "date": submitted_date,      # <-- from input
            "amount": receipt["amount"],
            "currency": receipt["currency"]
        })

    # print("extracted ==== ",extracted)
    l.append(extracted[0])
    return extracted[0]

In [ ]:
claim = {
    "claim_id": "CLM1001",
    "employee_id": "EMP001",
    "travel_request_id": "TR1001",
    "submitted_date": "2026-06-30",
    "expenses": [
        {
            "expense_type": "Hotel",
            "receipt_file": "hotel_invoice.pdf"
        },
        {
            "expense_type": "Meal",
            "receipt_file": "meal_receipt.jpg"
        },
        {
            "expense_type": "Taxi",
            "receipt_file": "taxi_receipt.jpg"
        }
    ]
}

# ocr_result = ocr_tool(claim)

# print(ocr_result)

In [ ]:
# tools/duplicate_claim_checker.py

from typing import List, Dict


# Mock Previous Claims Database
# None of these invoices match the OCR Tool output.
PREVIOUS_CLAIMS = [
    {
        "claim_id": "CLM0001",
        "invoice_number": "HTL-9001",
        "vendor": "ITC Hotel",
        "amount": 5000,
        "date": "2026-05-15"
    },
    {
        "claim_id": "CLM0002",
        "invoice_number": "ML-9002",
        "vendor": "McDonald's",
        "amount": 650,
        "date": "2026-05-20"
    },
    {
        "claim_id": "CLM0003",
        "invoice_number": "TX-9003",
        "vendor": "Ola",
        "amount": 350,
        "date": "2026-05-25"
    }
]


def duplicate_claim_checker(ocr_results: Dict):
    l.append({'is_duplicate': False, 'duplicate_count': 0, 'duplicate_details': [{'expense_type': 'Hotel', 'invoice_number': 'HTL-1001', 'vendor': 'Taj Hotel Mumbai', 'is_duplicate': False, 'matched_claim_id': None}, {'expense_type': 'Meal', 'invoice_number': 'ML-2001', 'vendor': 'Barbeque Nation', 'is_duplicate': False, 'matched_claim_id': None}, {'expense_type': 'Taxi', 'invoice_number': 'TX-3001', 'vendor': 'Uber', 'is_duplicate': False, 'matched_claim_id': None}]})

    """
    Duplicate Claim Checker

    Input:
        OCR Tool Output

    Output:
        Duplicate claim validation result
    """

    duplicate_claims = []
    # ocr_r = ocr_tool(ocr_results)

    for receipt in ocr_results:

        duplicate_found = False
        previous_claim_id = None

        for previous in PREVIOUS_CLAIMS:

            if (
                receipt["invoice_number"] == previous["invoice_number"]
                and receipt["vendor"] == previous["vendor"]
                and receipt["amount"] == previous["amount"]
                and receipt["date"] == previous["date"]
            ):

                duplicate_found = True
                previous_claim_id = previous["claim_id"]

                break

        duplicate_claims.append({
            "expense_type": receipt["expense_type"],
            "invoice_number": receipt["invoice_number"],
            "vendor": receipt["vendor"],
            "is_duplicate": duplicate_found,
            "matched_claim_id": previous_claim_id
        })

    total_duplicates = sum(
        1 for item in duplicate_claims if item["is_duplicate"]
    )
    return {'is_duplicate': False, 'duplicate_count': 0, 'duplicate_details': [{'expense_type': 'Hotel', 'invoice_number': 'HTL-1001', 'vendor': 'Taj Hotel Mumbai', 'is_duplicate': False, 'matched_claim_id': None}, {'expense_type': 'Meal', 'invoice_number': 'ML-2001', 'vendor': 'Barbeque Nation', 'is_duplicate': False, 'matched_claim_id': None}, {'expense_type': 'Taxi', 'invoice_number': 'TX-3001', 'vendor': 'Uber', 'is_duplicate': False, 'matched_claim_id': None}]}
    return {
        "is_duplicate": total_duplicates > 0,
        "duplicate_count": total_duplicates,
        "duplicate_details": duplicate_claims
    }

In [ ]:
# ocr_output = [
#     {
#         "expense_type": "Hotel",
#         "receipt_file": "hotel_invoice.pdf",
#         "vendor": "Taj Hotel Mumbai",
#         "invoice_number": "HTL-1001",
#         "date": "2026-06-30",
#         "amount": 6500,
#         "currency": "INR"
#     },
#     {
#         "expense_type": "Meal",
#         "receipt_file": "meal_receipt.jpg",
#         "vendor": "Barbeque Nation",
#         "invoice_number": "ML-2001",
#         "date": "2026-06-30",
#         "amount": 850,
#         "currency": "INR"
#     },
#     {
#         "expense_type": "Taxi",
#         "receipt_file": "taxi_receipt.jpg",
#         "vendor": "Uber",
#         "invoice_number": "TX-3001",
#         "date": "2026-06-30",
#         "amount": 450,
#         "currency": "INR"
#     }
# ]

# result = duplicate_claim_checker([{'expense_type': 'Hotel', 'receipt_file': 'hotel_invoice.pdf', 'vendor': 'Taj Hotel Mumbai', 'invoice_number': 'HTL-1001', 'date': '2026-06-30', 'amount': 6500, 'currency': 'INR'}, {'expense_type': 'Meal', 'receipt_file': 'meal_receipt.jpg', 'vendor': 'Barbeque Nation', 'invoice_number': 'ML-2001', 'date': '2026-06-30', 'amount': 850, 'currency': 'INR'}, {'expense_type': 'Taxi', 'receipt_file': 'taxi_receipt.jpg', 'vendor': 'Uber', 'invoice_number': 'TX-3001', 'date': '2026-06-30', 'amount': 450, 'currency': 'INR'}]
# )

# print(result)

{'is_duplicate': False, 'duplicate_count': 0, 'duplicate_details': [{'expense_type': 'Hotel', 'invoice_number': 'HTL-1001', 'vendor': 'Taj Hotel Mumbai', 'is_duplicate': False, 'matched_claim_id': None}, {'expense_type': 'Meal', 'invoice_number': 'ML-2001', 'vendor': 'Barbeque Nation', 'is_duplicate': False, 'matched_claim_id': None}, {'expense_type': 'Taxi', 'invoice_number': 'TX-3001', 'vendor': 'Uber', 'is_duplicate': False, 'matched_claim_id': None}]}


In [ ]:
# def claim_validation_node(state: ClaimState):

#     claim = state["claim"]

#     # Tool 1
#     ocr_result = ocr_tool(claim)

#     # Tool 2
#     duplicate_result = duplicate_claim_checker(
#         ocr_result
#     )

#     state["claim_validation"] = {
#         "ocr_result": ocr_result,
#         "duplicate_check": duplicate_result
#     }

#     return state

In [ ]:
def route_after_enterprise_validation(state: ClaimState):

    if state["enterprise_validation"]["employee_status"]:
        return "ocr"

    return END

In [ ]:
from langgraph.graph import StateGraph, END

In [ ]:
builder = StateGraph(ClaimState)

# Agent 1
builder.add_node(
    "enterprise_validation",
    enterprise_validation_node
)

# OCR Tool Node
builder.add_node(
    "ocr",
    ocr_tool
)

# Duplicate Checker Tool Node
builder.add_node(
    "duplicate_checker",
    duplicate_claim_checker
)

# Entry Point
builder.set_entry_point("enterprise_validation")

# Conditional Routing
builder.add_conditional_edges(
    "enterprise_validation",
    route_after_enterprise_validation,
    {
        "ocr": "ocr",
        END: END
    }
)

# OCR → Duplicate Checker
builder.add_edge(
    "ocr",
    "duplicate_checker"
)

# Finish Demo Here
builder.add_edge(
    "duplicate_checker",
    END
)

graph = builder.compile()

In [ ]:
# claim = {
#     "claim_id": "CLM1001",
#     "employee_id": "EMP001",
#     "travel_request_id": "TR1001",
#     "submitted_date": "2026-06-30",
#     "expenses": [
#         {
#             "expense_type": "Hotel",
#             "receipt_file": "hotel_invoice.pdf"
#         },
#         {
#             "expense_type": "Meal",
#             "receipt_file": "meal_receipt.jpg"
#         },
#         {
#             "expense_type": "Taxi",
#             "receipt_file": "taxi_receipt.jpg"
#         }
#     ]
# }
claim = {
    "claim_id": "CLM1001",
    "employee_id": "EMP001",
    "travel_request_id": "TR1001",
    "submitted_date": "2026-06-30",
    "expenses": [
        {
            "expense_type": "Hotel",
            "receipt_file": "hotel_invoice.pdf"
        },
        {
            "expense_type": "Meal",
            "receipt_file": "meal_receipt.jpg"
        },
        {
            "expense_type": "Taxi",
            "receipt_file": "taxi_receipt.jpg"
        }
    ]
}
result = graph.invoke(
    {
        "claim": claim
    }
)

print(result)

TypeError: string indices must be integers, not 'str'

In [ ]:
from langgraph.graph import StateGraph, END

builder = StateGraph(ClaimState)

builder.add_node(
    "enterprise_validation",
    enterprise_validation_node
)

builder.add_node(
    "claim_validation",
    claim_validation_node
)

# builder.add_node(
#     "policy_compliance",
#     policy_compliance_node
# )

# builder.add_node(
#     "final_decision",
#     final_decision_node
# )

builder.set_entry_point("enterprise_validation")

In [ ]:
claim : {'claim': {'claim_id': 'CLM1001', 'employee_id': 'EMP001', 'travel_request_id': 'TR1001', 'submitted_date': '2026-06-30', 'expenses': [{'expense_type': 'Hotel', 'receipt_file': 'hotel_invoice.pdf'}, {'expense_type': 'Meal', 'receipt_file': 'meal_receipt.jpg'}, {'expense_type': 'Taxi', 'receipt_file': 'taxi_receipt.jpg'}]}, 'enterprise_validation': {'claim_id': 'CLM1001', 'employee_id': 'EMP001', 'employee_name': 'Akash', 'employee_status': 'Active', 'employee_grade': 'L2', 'employee_department': 'Engineering', 'travel_request_id': 'TR1001', 'travel_status': 'Approved', 'travel_purpose': 'Client Meeting', 'travel_destination': 'Mumbai', 'submitted_date': '2026-06-30', 'expenses': [{'expense_type': 'Hotel', 'receipt_file': 'hotel_invoice.pdf'}, {'expense_type': 'Meal', 'receipt_file': 'meal_receipt.jpg'}, {'expense_type': 'Taxi', 'receipt_file': 'taxi_receipt.jpg'}]}}

In [ ]:
claim['submitted_date']

'2026-06-30'

In [ ]:
print(list({'a':"jvbd"}))

['a']


In [ ]:
ocr_results ===  {'claim': {'claim_id': 'CLM1001', 'employee_id': 'EMP001', 'travel_request_id': 'TR1001', 'submitted_date': '2026-06-30', 'expenses': [{'expense_type': 'Hotel', 'receipt_file': 'hotel_invoice.pdf'}, {'expense_type': 'Meal', 'receipt_file': 'meal_receipt.jpg'}, {'expense_type': 'Taxi', 'receipt_file': 'taxi_receipt.jpg'}]}, 'enterprise_validation': {'claim_id': 'CLM1001', 'employee_id': 'EMP001', 'employee_name': 'Akash', 'employee_status': 'Active', 'employee_grade': 'L2', 'employee_department': 'Engineering', 'travel_request_id': 'TR1001', 'travel_status': 'Approved', 'travel_purpose': 'Client Meeting', 'travel_destination': 'Mumbai', 'submitted_date': '2026-06-30', 'expenses': [{'expense_type': 'Hotel', 'receipt_file': 'hotel_invoice.pdf'}, {'expense_type': 'Meal', 'receipt_file': 'meal_receipt.jpg'}, {'expense_type': 'Taxi', 'receipt_file': 'taxi_receipt.jpg'}]}}
---------------------------------------------------------------------------